<a href="https://colab.research.google.com/github/ahlqui/VeloxChemColabs/blob/main/AtomicCharges.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>




# Calculate partial charges


This notebook requires that you have a molecular structure. Two options:

Define Molecule as XYZ-coordinates: Can be generatred with free software such as www.avogadro.cc

Define Molecule with SMILES code, A molecule can be defined using a SMILES code (example below). We have two suggested ways to generate smiles from structure. 1) Sketch your molecule at https://www.rcsb.org/chemical-sketch and the SMILES code will be shown right below the structure. 2) Build your molecule at https://molview.org/ , go to Tools/Information Card and it will show you the SMILES code. Just copy/paste it into the SMILES box below.

Example of xyz-coordinates for CO2:

O 0.00 0.00 0.00

C 0.00 0.00 1.20

O 0.00 0.00 2.40

Example of SMILES for CO2:

O=C=O

In [1]:
# Local environment check for QuimicaAutomocio P2.
# The original Colab version used a Colab-only installation cell.
# In local Jupyter, skip that installation cell and use the conda environment instead:
#   conda activate quimica-echem

import sys
try:
    import veloxchem as vlx
    print(f"VeloxChem imported correctly: {getattr(vlx, '__version__', 'unknown')}")
    print(f"Python executable: {sys.executable}")
except Exception as exc:
    raise RuntimeError(
        "VeloxChem is not available in this kernel. Activate the conda environment "
        "with `conda activate quimica-echem`, start Jupyter from that terminal, "
        "and select the quimica-echem kernel."
    ) from exc

VeloxChem imported correctly: 1.0rc4
Python executable: c:\Users\Nayanika\miniconda3\envs\quimica-echem\python.exe


In [2]:
xyz_coordinates = """
O             -3.346749868411        -2.868366048915        -0.075153307685
C             -3.187382591554        -1.708819916962         0.591286156306
O             -3.802237012621        -1.440460581721         1.593640691920
C             -2.100175247423        -0.842493813604        -0.015108866595
N             -2.390351399437         0.540869330053         0.288289183765
C             -0.726443743158        -1.382486798593         0.501579245412
C              0.432624575893        -0.582097690778        -0.028535724420
C              0.951519493650         0.511330571991         0.680970304319
C              1.979988556050         1.297940941174         0.164397577180
C              2.533560118514         1.018056290980        -1.099081908547
N              3.523008964671         1.823301512728        -1.641925894492
C              2.018215444984        -0.080702717966        -1.815448840033
C              0.990306562428        -0.856188111234        -1.286905461426
H             -4.009576149706        -3.384837467429         0.415328759844
H             -2.119272098748        -0.991926369757        -1.107814721085
H             -2.598648609166         0.639185757424         1.282418920138
H             -1.589991097211         1.130760931988         0.067633213559
H             -0.742446119366        -1.349692081979         1.603824438302
H             -0.632846916249        -2.439609813842         0.204957125734
H              0.549172746849         0.748143257251         1.669990967847
H              2.364651947294         2.140888174289         0.745370244370
H              4.060801011991         2.382848441180        -0.990857145140
H              4.088263067712         1.414181730254        -2.376289887802
H              2.433097081526        -0.323214363813        -2.797890836034
H              0.613015676361        -1.703097526418        -1.867880308776
"""


In [3]:
import veloxchem as vlx

molecule = vlx.Molecule.read_str(xyz_coordinates)
print('Structure of the molecule entered: ')
molecule.show(atom_indices=True)

Structure of the molecule entered: 


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [4]:
#@title DFT single point and charge calculation
#@markdown For classroom tests, start with a small basis such as STO-3G or DEF2-SVP.


#molecule.set_charge(1)

basis_set = 'DEF2-SVP' #@param  ['STO-3G', 'DEF2-SV(P)', 'DEF2-SVP', 'DEF2-SVPD', 'DEF2-TZVP']
basis = vlx.MolecularBasis.read(molecule, basis_set)
scf_drv = vlx.ScfRestrictedDriver()
method_dict = {'conv_thresh': 1e-4}
scf_drv.update_settings(method_dict)
mute_output = True # @param {type:"boolean"}
if mute_output:
    scf_drv.ostream.mute()
functional = 'B3LYP' #@param ['SLATER', 'BLYP', 'B3LYP', 'PBE', 'PBE0', 'BP86']
scf_drv.xcfun = functional
scf_results = scf_drv.compute(molecule, basis)

# VeloxChem 1.0 separates ESP and RESP drivers. Older Colab notebooks used
# RespChargesDriver(..., "esp"), which is now deprecated and raises an error.
esp_drv = vlx.EspChargesDriver()
esp_drv.ostream.mute()
esp_charges = esp_drv.compute(molecule, basis, scf_results)

resp_drv = vlx.RespChargesDriver()
resp_drv.ostream.mute()
resp_charges = resp_drv.compute(molecule, basis, scf_results)

chelpg_drv = vlx.EspChargesDriver()
chelpg_drv.ostream.mute()
chelpg_drv.grid_type = "chelpg"
chelpg_drv.equal_charges = "1=1, 1=1"
chelpg_charges = chelpg_drv.compute(molecule, basis, scf_results)

for title, charges in [
    ("ESP charge", esp_charges),
    ("RESP charge", resp_charges),
    ("CHELPG charge", chelpg_charges),
]:
    print(f"Atom      {title}")
    print(20 * "-")
    for label, charge in zip(molecule.get_labels(), charges):
        print(f"{label :s} {charge : 18.6f}")
    print(20 * "-")
    print(f"Total: {charges.sum() : 13.6f}\n")

Atom      ESP charge
--------------------
O          -0.604329
C           0.602605
O          -0.510068
C           0.418153
N          -0.960156
C          -0.324502
C           0.185955
C          -0.246704
C          -0.246961
C           0.335184
N          -0.727999
C          -0.265004
C          -0.184853
H           0.429822
H           0.014255
H           0.335565
H           0.379215
H           0.081406
H           0.075598
H           0.138428
H           0.145650
H           0.325947
H           0.329125
H           0.146731
H           0.126938
--------------------
Total:      0.000000

Atom      RESP charge
--------------------
O          -0.580527
C           0.574570
O          -0.495141
C           0.299234
N          -0.889627
C          -0.049475
C           0.028221
C          -0.217485
C          -0.153150
C           0.208832
N          -0.679189
C          -0.185907
C          -0.144052
H           0.420905
H           0.033235
H           0.319489
H          

In [5]:
#@title Visualize the charges on molecule
import py3Dmol
viewer = py3Dmol.view(linked=False, width=500, height=500)

charge_type = 'ESP' # @param ["RESP", "ESP", "CHELPG"]
charge_map = {
    'ESP': esp_charges,
    'RESP': resp_charges,
    'CHELPG': chelpg_charges,
}
charges = charge_map[charge_type]
print(f'{charge_type} charges:')
viewer.addModel(molecule.get_xyz_string(), "xyz")
for i, charge in enumerate(charges):
    viewer.addLabel(
        "{:.2f}".format(charge),
        {'backgroundOpacity': 0, 'fontColor': 'black', 'backgroundColor': 'white'},
        {'index': i},
    )
viewer.setStyle({"stick": {}, "sphere": {"radius": 0.4}})
viewer.rotate(-45, "y")
viewer.zoomTo()
viewer.show()

ESP charges:


3Dmol.js failed to load for some reason. Please check your browser console for error messages.